# 02 — UI Index Forensic Analysis

Interactive BAI/WBI/MIPI analysis with Plotly.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

df = pd.read_csv(ROOT / "data" / "dmv_macro_baselines.csv")
df["BAI"]  = df["Max_WBA"] / df["Weekly_Housing"]
df["WBI"]  = df["Taxable_Wage_Base"] / df["Avg_Annual_Wage"]
df["MIPI"] = max(0, 250 - 50) / df["Max_WBA"]
df["Housing_Gap"] = df["Weekly_Housing"] - df["Max_WBA"]

COLORS = {"Maryland": "#4aa8d8", "Virginia": "#00FF41", "District of Columbia": "#D4AF37"}
LAYOUT = dict(paper_bgcolor="#121212", plot_bgcolor="#1e1e1e",
              font=dict(color="#e8e8e8", family="monospace"),
              legend=dict(bgcolor="#1e1e1e"))
print(df.to_string())


## Benefit Adequacy Index (BAI) — 2010→2026

In [ ]:
fig = go.Figure()
for jur, color in COLORS.items():
    sub = df[df["Jurisdiction"] == jur].sort_values("Year")
    fig.add_trace(go.Scatter(x=sub["Year"], y=sub["BAI"], name=jur,
                              mode="lines+markers+text",
                              line=dict(color=color, width=2.5), marker=dict(size=9),
                              text=[f"{v:.2f}" for v in sub["BAI"]],
                              textposition="top right", textfont=dict(color=color)))
fig.add_hline(y=1.0, line_dash="dash", line_color="#DC143C",
              annotation_text="Survival Threshold (1.0)")
fig.update_layout(**LAYOUT, title="BAI Decay Trajectory: 2010 → 2026",
                  xaxis=dict(tickvals=[2010, 2018, 2026]),
                  yaxis=dict(range=[0.5, 1.7], title="BAI"))
fig.show()


## Sensitivity Analysis — What if wage base tracked inflation?

In [ ]:
# Load CPI data if available
import json
cpi_path = ROOT / "data" / "cpi_annual.json"
if cpi_path.exists():
    with open(cpi_path) as f:
        cpi_cache = json.load(f)
    cpi = {int(yr): val for yr, val in cpi_cache["data"].items()}
    print("CPI data available:")
    for yr in [2010, 2014, 2018, 2023]:
        print(f"  {yr}: {cpi.get(yr, 'N/A')}")
else:
    print("CPI cache not found — run fetch_bls_baselines.py")
    cpi = {2010: 218.1, 2018: 251.1, 2023: 304.7}

# Counterfactual: what would MD benefit cap be if indexed to CPI from 2010?
md_2010_wba = 430
for yr, cpi_val in sorted(cpi.items()):
    if yr >= 2010:
        indexed_wba = md_2010_wba * (cpi_val / cpi[2010])
        print(f"  MD CPI-indexed WBA {yr}: ${indexed_wba:,.0f}")


## Housing Gap Divergence

In [ ]:
fig2 = make_subplots(rows=1, cols=3, subplot_titles=["Maryland", "Virginia", "DC"])
for i, jur in enumerate(["Maryland", "Virginia", "District of Columbia"], 1):
    sub = df[df["Jurisdiction"] == jur].sort_values("Year")
    color = COLORS[jur]
    fig2.add_trace(go.Scatter(x=sub["Year"], y=sub["Weekly_Housing"],
                               name="Housing", line=dict(color="#DC143C", width=2.5),
                               showlegend=(i==1)), row=1, col=i)
    fig2.add_trace(go.Scatter(x=sub["Year"], y=sub["Max_WBA"],
                               name="Max WBA", line=dict(color=color, width=2.5, dash="dash"),
                               showlegend=(i==1)), row=1, col=i)
fig2.update_layout(**LAYOUT, title="Housing Cost vs. Max WBA — The Survival Deficit")
fig2.show()
